In [12]:
# ============================================================
# RETAILPULSE - SECTION 5
# INVENTORY OPTIMIZATION
# MODEL: HOLT EXPONENTIAL SMOOTHING
# ============================================================

import pandas as pd
import numpy as np
import warnings
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# CONFIG
FORECAST_DAYS = 30
LEAD_TIME_DAYS = 7
SERVICE_LEVEL_Z = 1.65
MIN_HISTORY_DAYS = 20
STOCK_COVERAGE_DAYS = 30
OUTPUT_PATH = "../data/processed/inventory_optimization_holt.csv"

print("=" * 60)
print("RETAILPULSE - INVENTORY OPTIMIZATION")
print("MODEL: HOLT EXPONENTIAL SMOOTHING")
print("=" * 60)

# LOAD DATA
df = pd.read_csv("../data/processed/cleaned_retail.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df = df.dropna(subset=["InvoiceDate", "StockCode", "Quantity"])

# DAILY DEMAND
daily_sales = (
    df.assign(Date=df["InvoiceDate"].dt.floor("D"))
    .groupby(["StockCode", "Date"], as_index=False)["Quantity"].sum()
    .rename(columns={"StockCode": "Product", "Date": "ds", "Quantity": "y"})
)

total_products = daily_sales["Product"].nunique()

# FORECASTING
product_forecasts, product_statistics = [], []
successful_holt, fallback_ma, insufficient_history = 0, 0, 0

for i, (product, product_data) in enumerate(daily_sales.groupby("Product"), start=1):
    product_data = product_data[["ds", "y"]].sort_values("ds")
    complete_dates = pd.date_range(product_data["ds"].min(), product_data["ds"].max(), freq="D")

    sales_series = (
        product_data.set_index("ds")["y"]
        .reindex(complete_dates, fill_value=0)
        .astype(float)
    )

    if len(sales_series) < MIN_HISTORY_DAYS:
        insufficient_history += 1
        continue

    mean_daily_demand = float(sales_series.mean())
    std_daily_demand = float(sales_series.std(ddof=1)) if not np.isnan(sales_series.std(ddof=1)) else 0.0

    forecast = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model = Holt(sales_series, initialization_method="estimated")
            fitted_model = model.fit(optimized=True)
            forecast = fitted_model.forecast(FORECAST_DAYS)

        forecast = np.asarray(forecast, dtype=float)
        if len(forecast) == FORECAST_DAYS and np.all(np.isfinite(forecast)):
            successful_holt += 1
        else:
            forecast = None
    except Exception:
        forecast = None

    if forecast is None:
        fallback_ma += 1
        recent_window = sales_series.tail(min(30, len(sales_series)))
        average_demand = float(recent_window.mean())
        forecast = np.repeat(average_demand, FORECAST_DAYS)

    forecast = np.maximum(forecast, 0)
    forecast_dates = pd.date_range(complete_dates.max() + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq="D")
    forecast_out = pd.DataFrame({"Product": product, "ds": forecast_dates, "ForecastedDemand": forecast})
    product_forecasts.append(forecast_out)

    product_statistics.append({
        "Product": product,
        "MeanDailyDemand": mean_daily_demand,
        "StdDailyDemand": std_daily_demand,
        "HistoricalDays": len(sales_series)
    })

# VALIDATION
if len(product_forecasts) == 0:
    raise ValueError("No products had sufficient historical demand.")

forecast_df = pd.concat(product_forecasts, ignore_index=True)
statistics_df = pd.DataFrame(product_statistics)

# INVENTORY CALCULATIONS
forecast_summary = forecast_df.groupby("Product", as_index=False)["ForecastedDemand"].sum().rename(columns={"ForecastedDemand":"ForecastedDemand30D"})
lead_time_demand = (
    forecast_df.sort_values(["Product","ds"])
    .groupby("Product").head(LEAD_TIME_DAYS)
    .groupby("Product", as_index=False)["ForecastedDemand"].sum()
    .rename(columns={"ForecastedDemand":"LeadTimeDemand"})
)

inventory_df = (
    statistics_df.merge(forecast_summary, on="Product", how="left")
    .merge(lead_time_demand, on="Product", how="left")
)

inventory_df["LeadTimeDays"] = LEAD_TIME_DAYS
inventory_df["SafetyStock"] = SERVICE_LEVEL_Z * inventory_df["StdDailyDemand"] * np.sqrt(LEAD_TIME_DAYS)
inventory_df["EstimatedInventoryProxy"] = inventory_df["MeanDailyDemand"] * STOCK_COVERAGE_DAYS

inventory_df["ReorderPoint"] = inventory_df["LeadTimeDemand"] + inventory_df["SafetyStock"]
inventory_df["ReorderQty"] = (inventory_df["ReorderPoint"] - inventory_df["EstimatedInventoryProxy"]).clip(lower=0)
inventory_df["InventoryStatus"] = np.where(inventory_df["EstimatedInventoryProxy"] < inventory_df["ReorderPoint"], "REORDER", "SUFFICIENT")

numeric_columns = ["MeanDailyDemand","StdDailyDemand","ForecastedDemand30D","LeadTimeDemand","SafetyStock","EstimatedInventoryProxy","ReorderPoint","ReorderQty"]
inventory_df[numeric_columns] = inventory_df[numeric_columns].round(2)

# SAVE OUTPUT
inventory_df.to_csv(OUTPUT_PATH, index=False)

print("=" * 60)
print("INVENTORY OPTIMIZATION COMPLETED")
print("=" * 60)
print("Products analyzed:", inventory_df["Product"].nunique())
print("Total recommended reorder quantity:", round(inventory_df["ReorderQty"].sum(), 2))
print("Products requiring reorder:", (inventory_df["InventoryStatus"]=="REORDER").sum())
print("Output saved to:", OUTPUT_PATH)


RETAILPULSE - INVENTORY OPTIMIZATION
MODEL: HOLT EXPONENTIAL SMOOTHING
INVENTORY OPTIMIZATION COMPLETED
Products analyzed: 4447
Total recommended reorder quantity: 220617.28
Products requiring reorder: 2210
Output saved to: ../data/processed/inventory_optimization_holt.csv
